# 00 · Data check: ATLAS Higgs ML Challenge 2014

Source: CERN Open Data record 328 (https://opendata.cern.ch/record/328), file `atlas-higgs-challenge-2014-v2.csv.gz` (65,630,848 bytes, adler32 `e4717bbe`), saved in `data/` (gitignored). This notebook loads it with `src.data.load`, checks its shape, class balance and `-999` placeholders, lists the features `select_features` keeps, and writes `results/data_summary.json`.

In [1]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src import data

PATH = ROOT / 'data' / 'atlas-higgs-challenge-2014-v2.csv.gz'
df = data.load(str(PATH))
raw_cols = [c for c in df.columns if c != 'y']  # data.load adds the 0/1 label column 'y'
print(f'{len(df):,} rows x {len(raw_cols)} columns in the file (+ y added by data.load)')

818,238 rows x 35 columns in the file (+ y added by data.load)


## Columns
`PRI_*` are primary quantities measured by the detector, `DER_*` are derived from them by physicists. `EventId`, `Weight`, `Label`, `KaggleSet` and `KaggleWeight` are bookkeeping, not features.

In [2]:
feats = data.feature_columns(df)
print(len(feats), 'features:', sum(c.startswith('PRI_') for c in feats), 'PRI +', sum(c.startswith('DER_') for c in feats), 'DER')
print('non-feature columns:', [c for c in raw_cols if c not in feats])
print('KaggleSet:', df['KaggleSet'].value_counts().to_dict())

30 features: 17 PRI + 13 DER
non-feature columns: ['EventId', 'Weight', 'Label', 'KaggleSet', 'KaggleWeight']
KaggleSet: {'v': 450000, 't': 250000, 'b': 100000, 'u': 18238}


## Signal fraction
Unweighted = share of rows labelled `s`. The simulation deliberately over-produces signal; the physics `Weight` column scales events back to their real expected rates, so the weighted fraction is the realistic one.

In [3]:
n_sig = int(df['y'].sum())
sig_frac = n_sig / len(df)
w_sig_frac = df.loc[df.y == 1, 'Weight'].sum() / df['Weight'].sum()
print(f'signal events: {n_sig:,} / {len(df):,}  ->  unweighted fraction {sig_frac:.4f}')
print(f'weighted signal fraction (Weight column): {w_sig_frac:.5f}')

signal events: 279,560 / 818,238  ->  unweighted fraction 0.3417
weighted signal fraction (Weight column): 0.00168


## Features containing -999
Fraction of all events where each feature is the `-999.0` placeholder, also split by class.

In [4]:
is_missing = df[feats] == data.MISSING
miss = pd.DataFrame({
    'fraction_all': is_missing.mean(),
    'fraction_background': is_missing[df.y == 0].mean(),
    'fraction_signal': is_missing[df.y == 1].mean(),
})
miss = miss[miss.fraction_all > 0].sort_values('fraction_all')
miss.round(4)

,fraction_all,fraction_background,fraction_signal
DER_mass_MMC,0.1523,0.2143,0.0329
PRI_jet_leading_phi,0.4001,0.4539,0.2964
PRI_jet_leading_eta,0.4001,0.4539,0.2964
PRI_jet_leading_pt,0.4001,0.4539,0.2964
DER_prodeta_jet_jet,0.7091,0.7559,0.6190
DER_mass_jet_jet,0.7091,0.7559,0.6190
DER_deltaeta_jet_jet,0.7091,0.7559,0.6190
DER_lep_eta_centrality,0.7091,0.7559,0.6190
PRI_jet_subleading_pt,0.7091,0.7559,0.6190
PRI_jet_subleading_eta,0.7091,0.7559,0.6190


### Is the missingness structural?
Jet quantities only exist if the event has enough jets. Check that each jet feature is `-999` **exactly** when `PRI_jet_num` is too small; `DER_mass_MMC` is the exception.

In [5]:
rule = {c: 1 for c in ['PRI_jet_leading_pt', 'PRI_jet_leading_eta', 'PRI_jet_leading_phi']}
rule.update({c: 2 for c in ['PRI_jet_subleading_pt', 'PRI_jet_subleading_eta', 'PRI_jet_subleading_phi',
                            'DER_deltaeta_jet_jet', 'DER_mass_jet_jet', 'DER_prodeta_jet_jet',
                            'DER_lep_eta_centrality']})
structural = {}
for c, need in rule.items():
    structural[c] = bool((is_missing[c] == (df['PRI_jet_num'] < need)).all())
    print(f'{c:26s} -999  <=>  PRI_jet_num < {need}: {structural[c]}')
assert set(rule) | {'DER_mass_MMC'} == set(miss.index)
print()
print('events by PRI_jet_num:', df['PRI_jet_num'].value_counts().sort_index().to_dict())
print('DER_mass_MMC missing vs jet count:',
      is_missing.groupby(df['PRI_jet_num'])['DER_mass_MMC'].mean().round(3).to_dict())

PRI_jet_leading_pt         -999  <=>  PRI_jet_num < 1: True
PRI_jet_leading_eta        -999  <=>  PRI_jet_num < 1: True
PRI_jet_leading_phi        -999  <=>  PRI_jet_num < 1: True
PRI_jet_subleading_pt      -999  <=>  PRI_jet_num < 2: True
PRI_jet_subleading_eta     -999  <=>  PRI_jet_num < 2: True
PRI_jet_subleading_phi     -999  <=>  PRI_jet_num < 2: True
DER_deltaeta_jet_jet       -999  <=>  PRI_jet_num < 2: True
DER_mass_jet_jet           -999  <=>  PRI_jet_num < 2: True
DER_prodeta_jet_jet        -999  <=>  PRI_jet_num < 2: True
DER_lep_eta_centrality     -999  <=>  PRI_jet_num < 2: True

events by PRI_jet_num: {0: 327371, 1: 252882, 2: 165027, 3: 72958}
DER_mass_MMC missing vs jet count: {0: 0.261, 1: 0.098, 2: 0.059, 3: 0.065}


## Features kept by `select_features`
Always-defined features only (none of the above), raw `*_phi` angles dropped, and the three rotation-invariant `|Δφ|` differences added.

In [6]:
df_sel, cols = data.select_features(df)
print(len(cols), 'features:')
for c in cols:
    print(' ', c)
assert not (df_sel[cols] == data.MISSING).any().any()

19 features:
  DER_mass_transverse_met_lep
  DER_mass_vis
  DER_pt_h
  DER_deltar_tau_lep
  DER_pt_tot
  DER_sum_pt
  DER_pt_ratio_lep_tau
  DER_met_phi_centrality
  PRI_tau_pt
  PRI_tau_eta
  PRI_lep_pt
  PRI_lep_eta
  PRI_met
  PRI_met_sumet
  PRI_jet_num
  PRI_jet_all_pt
  DPHI_tau_lep
  DPHI_tau_met
  DPHI_lep_met


In [7]:
summary = {
    'source': 'https://opendata.cern.ch/record/328',
    'file': PATH.name, 'file_bytes': PATH.stat().st_size, 'adler32': 'e4717bbe',
    'n_rows': len(df), 'n_columns': len(raw_cols), 'columns': raw_cols,
    'n_features_raw': len(feats),
    'n_signal': n_sig, 'n_background': len(df) - n_sig,
    'signal_fraction': sig_frac, 'signal_fraction_weighted': float(w_sig_frac),
    'kaggle_set_counts': {k: int(v) for k, v in df['KaggleSet'].value_counts().items()},
    'jet_num_counts': {int(k): int(v) for k, v in df['PRI_jet_num'].value_counts().sort_index().items()},
    'missing_value': data.MISSING,
    'missing_fraction': {c: {k: float(v) for k, v in r.items()} for c, r in miss.iterrows()},
    'missing_is_structural_jet_num': structural,
    'selected_features': cols, 'n_selected_features': len(cols),
}
out = ROOT / 'results' / 'data_summary.json'
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(summary, indent=2) + '\n')
print('wrote', out.relative_to(ROOT))

wrote results\data_summary.json


## In plain words

**-999 means "does not exist for this event", not "lost".** Jet quantities need jets: an event with 0 jets has no leading jet (40% of events), and anything built from *two* jets (dijet mass, Δη between jets, the lepton's position between the jets) needs at least 2 (missing in 71%). The check above shows this is exact. The model must not read -999 as a real energy or angle, so `select_features` keeps only the 19 features that are always defined.

**DER_mass_MMC** is the best estimate of the mass of the particle that decayed into the τ lepton and hadronic τ. Neutrinos escape unseen, so the mass can't be measured directly; the Missing Mass Calculator scans the possible neutrino momenta and picks the most likely mass given the detector's resolution. A real Higgs gives a peak near 125 GeV, which makes this the single most powerful feature. When no neutrino solution fits the event, the calculator fails and writes -999 (15% of events). That failure is itself informative: it happens in 21% of background but only 3% of signal events. It is excluded here; `select_features(mmc_flag=True)` can add a 0/1 "MMC defined" flag instead.

**Signal fraction:** 34% of rows are signal, because the simulation over-produced Higgs events on purpose. Weighted to real LHC rates (`Weight`), signal is only ~0.17%, i.e. about 1 Higgs event per 600 background events. The study uses balanced train/test samples and ROC-AUC, which don't depend on that ratio.